#EXERCICE 1

1. Énoncé clair du problème

L’objectif de ce projet est de développer un modèle d’apprentissage automatique binaire supervisé capable de prédire si un demandeur de prêt est susceptible de faire défaut (c’est-à-dire de ne pas rembourser son prêt conformément aux termes convenus) avant l’octroi du prêt. À partir d’un ensemble de caractéristiques relatives au demandeur (historique financier, situation professionnelle, caractéristiques du prêt, etc.), le modèle devra produire une probabilité de défaut. Cette prédiction permettra à l’institution financière de prendre des décisions éclairées concernant l’approbation du prêt, la fixation du taux d’intérêt et la détermination du montant accordé, réduisant ainsi les pertes financières tout en gérant le risque de crédit. Le critère de succès est le suivant : le modèle devra atteindre un rappel (recall) élevé pour la classe « défaut » (afin d’identifier le plus possible de mauvais payeurs) tout en maintenant une précision (precision) et une aire sous la courbe ROC (AUC-ROC) satisfaisantes, selon un seuil de décision ajustable en fonction de l’appétit pour le risque de l’établissement.

2. Types de données nécessaires

Pour une prédiction efficace des défauts, les données peuvent être regroupées en plusieurs catégories. La première catégorie concerne les informations personnelles et démographiques : âge, ancienneté dans l’adresse actuelle, situation de famille, nombre de personnes à charge (sous réserve des considérations légales et éthiques). La deuxième catégorie porte sur la situation professionnelle et les revenus : statut d’emploi (CDI, indépendant, etc.), ancienneté dans l’emploi, revenu annuel, ratio dette/revenu. La troisième catégorie couvre l’historique financier et le crédit : score de crédit (FICO, etc.), nombre de comptes ouverts, utilisation du crédit (taux d’endettement), antécédents de retards de paiement, nombre de défauts antérieurs, faillites. La quatrième catégorie regroupe les caractéristiques du prêt demandé : montant du prêt, durée du prêt (en mois), type de prêt (personnel, auto, hypothécaire), taux d’intérêt proposé, ratio montant du prêt / valeur de la garantie (LTV). Enfin, une cinquième catégorie peut inclure, si disponibles, des données comportementales : historique des transactions bancaires, fréquence d’utilisation des découverts, habitudes d’épargne, paiements de factures récurrentes (loyer, services publics). La variable cible (label) est définie ainsi : 0 pour un remboursement normal (non-défaut) et 1 pour un défaut de paiement (généralement défini comme un retard supérieur à 90 jours ou une absence de remboursement selon les politiques internes).

3. Sources potentielles de données

Les données internes à l’institution financière constituent une première source essentielle : systèmes de gestion des prêts (historique des prêts antérieurs, statuts de remboursement), bases de données clients (informations démographiques, relevés bancaires internes), journaux de transactions et comptes courants. Les agences d’évaluation du crédit comme Equifax, Experian ou TransUnion fournissent les scores de crédit, les rapports de crédit et les historiques de paiement. Des sources externes ouvertes ou payantes peuvent être mobilisées : registres publics (faillites, saisies, jugements), données macroéconomiques (taux de chômage local, PIB, inflation – pour le contexte), données de revenus vérifiés via services de vérification d’emploi ou déclarations fiscales par API. Enfin, des données alternatives issues de la fintech ou de startups peuvent être envisagées : données de téléphonie mobile (régularité des recharges), paiements de loyers et services publics via des agences comme Rental Kharma. Une remarque légale et éthique s’impose : toute collecte doit respecter les réglementations en vigueur (RGPD, loi Informatique et Libertés, Equal Credit Opportunity Act aux États-Unis). Il est interdit d’utiliser des caractéristiques sensibles (race, religion, orientation sexuelle) ou d’introduire des biais discriminatoires.

4. Plan de collecte de données

La première étape consiste à accéder aux données historiques internes en extrayant un échantillon des 3 à 5 dernières années de dossiers de prêts (incluant à la fois les prêts remboursés et ceux en défaut), et en s’assurant que la variable cible (défaut) est correctement définie et étiquetée. La deuxième étape est l’enrichissement via les agences de crédit : il faut signer des accords de partage avec une ou plusieurs agences pour obtenir le score de crédit et l’historique complet des demandeurs (avec consentement explicite des clients). La troisième étape est l’intégration des données démographiques et professionnelles : collecter les données via les formulaires de demande de prêt (sous contrôle des règles de conformité), puis nettoyer les valeurs manquantes (exemple : revenu non renseigné) et standardiser les formats. La quatrième étape consiste à ajouter des variables dérivées : calculer le ratio dette/revenu (total des paiements mensuels divisé par revenu mensuel), calculer le ratio montant du prêt sur valeur de la garantie (LTV) pour les prêts hypothécaires, générer des indicateurs temporels (ancienneté dans l’emploi, âge du compte bancaire). La cinquième étape porte sur les considérations d’échantillonnage : la classe « défaut » étant souvent minoritaire (déséquilibre de classes), il faut prévoir un sur-échantillonnage (SMOTE) ou une pondération lors de la modélisation, et s’assurer que l’échantillon couvre différentes périodes économiques (récession, croissance) pour la robustesse. Enfin, la sixième étape concerne le stockage et la gouvernance : centraliser les données dans un entrepôt de données sécurisé (exemple : BigQuery, Redshift) avec pseudonymisation des identifiants personnels, et mettre en place des audits réguliers pour détecter les biais potentiels. Ce document constitue le livrable de l’exercice 1 et servira de base pour les étapes suivantes : sélection des caractéristiques, choix des modèles et évaluation des performances.

In [ ]:
#EXERCICE 2



#EXERCICE 3

1. Choix des modèles pour la prédiction de prêt

Pour un problème de prédiction de défaut de paiement, qui est un problème de classification binaire supervisée avec un fort déséquilibre de classes (peu de défauts par rapport aux non-défauts) et un besoin d’interprétabilité pour des raisons réglementaires, je choisirais trois types de modèles complémentaires.

Le premier modèle est la régression logistique. Elle est simple, rapide à entraîner, très interprétable (elle fournit des coefficients pour chaque caractéristique, permettant d’expliquer pourquoi un demandeur est jugé risqué) et constitue une excellente baseline. Elle fonctionne bien lorsque les relations entre les variables et la cible sont approximativement linéaires.

Le deuxième modèle est la forêt aléatoire (Random Forest) . Il s’agit d’une méthode d’ensemble basée sur des arbres de décision. Elle capture naturellement les non-linéarités et les interactions entre caractéristiques (par exemple, l’effet combiné d’un faible revenu et d’un montant de prêt élevé). Elle est robuste aux valeurs aberrantes et aux caractéristiques corrélées, et elle fournit des mesures d’importance des variables, ce qui aide à la sélection des caractéristiques.

Le troisième modèle est XGBoost (eXtreme Gradient Boosting) . C’est l’un des modèles les plus performants pour les données tabulaires structurées comme celles des prêts. Il optimise de manière itérative les erreurs du modèle précédent, gère bien le déséquilibre des classes via des paramètres comme scale_pos_weight, et offre de nombreuses possibilités de réglage fin (régularisation L1/L2, taux d’apprentissage, profondeur des arbres). Il est cependant moins interprétable directement, mais on peut utiliser SHAP pour expliquer ses prédictions.

En pratique, je recommande de commencer par la régression logistique comme baseline, puis d’expérimenter avec Random Forest et XGBoost pour améliorer les performances, tout en conservant un œil sur l’interprétabilité.

2. Étapes d’évaluation des performances du modèle

L’évaluation se déroule en plusieurs étapes méthodiques.

Première étape : partitionnement rigoureux des données. Avant tout entraînement, je divise l’ensemble des données en trois sous-ensembles : un ensemble d’entraînement (70 %), un ensemble de validation (15 %) pour le réglage des hyperparamètres, et un ensemble de test (15 %) qui ne sera utilisé qu’une seule fois à la fin pour estimer la performance réelle en production. Il est crucial de préserver la distribution temporelle (par exemple, entraîner sur des prêts plus anciens et tester sur des prêts plus récents) pour éviter le biais de temps et refléter la réalité opérationnelle.

Deuxième étape : gestion du déséquilibre des classes. Comme la classe « défaut » est minoritaire (souvent 1 à 5 % des cas), j’applique des techniques adaptées pendant l’entraînement : sur-échantillonnage de la classe minoritaire avec SMOTE (Synthetic Minority Over-sampling Technique), sous-échantillonnage de la classe majoritaire, ou utilisation de poids de classe inversement proportionnels aux fréquences. Je n’applique jamais ces transformations sur l’ensemble de test ; seul l’ensemble d’entraînement est modifié.

Troisième étape : choix des métriques d’évaluation pertinentes. Dans ce contexte financier, l’accuracy (précision globale) est trompeuse : un modèle qui prédit « non-défaut » pour tout le monde aurait une accuracy de 95 à 99 % mais serait totalement inutile. Je retiens donc les métriques suivantes. Le rappel (recall) pour la classe défaut (aussi appelé sensibilité ou taux de vrais positifs) est la métrique la plus importante : elle mesure la proportion de véritables défauts que le modèle a correctement identifiés. Un rappel élevé signifie que peu de mauvais payeurs passent à travers les mailles du filet. La précision (precision) pour la classe défaut mesure, parmi tous les dossiers prédits comme « défaut », combien le sont réellement. Une précision faible entraînerait trop de refus de prêt à des clients solvables. L’aire sous la courbe ROC (AUC-ROC) résume la capacité du modèle à distinguer les deux classes indépendamment du seuil de décision. L’aire sous la courbe précision-rappel (AUC-PR) est particulièrement adaptée aux datasets déséquilibrés et complète l’AUC-ROC. Enfin, le score F1 (moyenne harmonique de la précision et du rappel) offre un équilibre utile.

Quatrième étape : validation croisée et réglage des hyperparamètres. J’utilise une validation croisée stratifiée à 5 plis sur l’ensemble d’entraînement pour obtenir des estimations stables des performances. Pour chaque modèle, je recherche les meilleurs hyperparamètres via une recherche par grille (Grid Search) ou aléatoire (Random Search), en optimisant par exemple le rappel ou l’AUC-ROC selon l’objectif métier. Pour XGBoost, je fais varier la profondeur des arbres, le taux d’apprentissage, le nombre d’estimateurs et le paramètre d’équilibrage des classes.

Cinquième étape : évaluation finale sur l’ensemble de test. Une fois le meilleur modèle sélectionné sur l’ensemble de validation, je l’exécute une seule fois sur l’ensemble de test pour estimer sa performance en conditions réelles. Je calcule alors toutes les métriques (rappel, précision, AUC-ROC, AUC-PR, F1) ainsi que la matrice de confusion. Une attention particulière est portée au coût métier : je détermine le seuil de décision optimal non pas à 0,5 par défaut, mais en maximisant un gain financier simulé (par exemple, coût d’un défaut non détecté versus coût d’un refus de prêt à un bon client).

Sixième étape : analyse des erreurs et optimisation itérative. J’examine les faux négatifs (défauts non détectés) et les faux positifs (bons clients refusés à tort) pour identifier des schémas. Si les faux négatifs concernent des profils particuliers (par exemple, jeunes travailleurs avec peu d’historique de crédit), cela peut orienter la collecte de nouvelles caractéristiques. Enfin, je compare la stabilité des modèles : un modèle trop complexe (XGBoost très profond) peut être performant sur le test mais instable dans le temps. Pour une application bancaire, je privilégie un compromis entre performance et robustesse, en testant également le modèle sur des données d’une année différente (backtesting temporel).

#EXERCICE 4

Scénario 1 : Prévision des cours boursiers – prédire les cours futurs

Pour la prévision des cours boursiers, le type d’apprentissage automatique le plus approprié est l’apprentissage supervisé, plus précisément la régression (car il s’agit de prédire une valeur numérique continue – le prix futur). Cependant, ce problème présente des particularités qui orientent vers des modèles séquentiels. Les cours boursiers sont des séries temporelles, ce qui signifie que l’ordre temporel des observations est essentiel et qu’il existe une dépendance entre une valeur future et ses valeurs passées. Je choisirais donc des modèles spécialisés pour les séquences : les réseaux de neurones récurrents (RNN), notamment les variantes LSTM (Long Short-Term Memory) ou GRU (Gated Recurrent Unit), qui sont capables de mémoriser des dépendances à long terme. Les modèles plus classiques comme ARIMA (AutoRegressive Integrated Moving Average) relèvent davantage de la statistique, mais en apprentissage automatique pur, on peut aussi utiliser des forêts aléatoires ou XGBoost avec des caractéristiques dérivées (retards, moyennes mobiles, indicateurs techniques). L’apprentissage par renforcement peut également être envisagé si l’objectif n’est pas simplement de prédire le prix mais de prendre des décisions d’achat/vente successives. Dans tous les cas, la prédiction boursière est extrêmement difficile car les marchés sont bruités, non stationnaires et influencés par des événements imprévisibles. On utilise donc généralement un apprentissage supervisé sur des fenêtres glissantes d’historique, avec validation temporelle stricte (pas de mélange aléatoire des données) et des métriques comme l’erreur quadratique moyenne (MSE) ou l’erreur absolue moyenne (MAE). Justification : la tâche est clairement définie avec des données étiquetées (prix passé connu, prix futur à prédire), et la dimension temporelle impose de respecter l’ordre chronologique.

Scénario 2 : Organiser une bibliothèque – regrouper les livres par genres ou catégories en fonction de leurs similitudes

Pour organiser une bibliothèque en regroupant les livres par genres ou catégories sans disposer d’étiquettes préexistantes, le type d’apprentissage automatique le plus approprié est l’apprentissage non supervisé, plus précisément le clustering (regroupement). Il n’y a pas de variable cible à prédire ; l’objectif est de découvrir des structures naturelles dans les données en se basant uniquement sur les caractéristiques des livres (titre, résumé, mots-clés, couverture, etc.). Plusieurs algorithmes de clustering peuvent être utilisés. K-means est efficace si l’on connaît approximativement le nombre souhaité de genres (par exemple, 10 ou 20 catégories). DBSCAN (Density-Based Spatial Clustering of Applications with Noise) a l’avantage de ne pas imposer de nombre fixe de clusters et de pouvoir identifier des livres « atypiques » qui n’appartiennent à aucun genre clair. Pour des textes, on peut aussi utiliser des méthodes hiérarchiques (agglomératives) qui produisent un dendrogramme et permettent une exploration interactive. Une approche plus moderne consisterait à transformer chaque livre en un vecteur numérique via un modèle de type word embeddings (Word2Vec, GloVe) ou mieux, via un transformer comme BERT ou Sentence-BERT pour capturer le sens profond du texte. Ensuite, on applique un algorithme de clustering sur ces vecteurs. Justification : l’absence d’étiquettes (pas de liste préétablie des genres pour chaque livre) rend l’apprentissage supervisé impossible. L’objectif est de créer des groupes homogènes où les livres d’un même cluster sont plus similaires entre eux qu’avec ceux d’autres clusters. Ce type d’approche permet également de révéler des genres émergents ou des catégories transversales que des étiquettes humaines auraient pu ignorer.

Scénario 3 : Programmer un robot pour qu’il navigue et trouve le chemin le plus court dans un labyrinthe

Pour qu’un robot apprenne à naviguer et à trouver le chemin le plus court dans un labyrinthe, le type d’apprentissage automatique le plus approprié est l’apprentissage par renforcement (reinforcement learning). Dans ce cadre, un agent (le robot) interagit avec un environnement (le labyrinthe), effectue des actions (avancer, tourner à gauche, tourner à droite, etc.) et reçoit des récompenses (positive lorsqu’il atteint la sortie, négative s’il heurte un mur ou perd du temps). L’objectif est d’apprendre une politique (policy) qui maximise la récompense cumulée sur le long terme, ce qui conduit naturellement au chemin le plus court sans nécessiter d’exemples pré-étiquetés de chemins optimaux. Plusieurs algorithmes de reinforcement learning sont possibles selon la complexité du labyrinthe. Pour un labyrinthe simple avec des états discrets (cases), le Q-learning tabulaire est un excellent point de départ : il apprend une table des valeurs Q pour chaque paire (état, action). Pour des labyrinthes plus grands ou continus, on utilise le Deep Q-Network (DQN) qui approxime la fonction Q avec un réseau de neurones. On peut aussi utiliser des méthodes de gradient de politique comme PPO (Proximal Policy Optimization). Justification : contrairement à l’apprentissage supervisé, on ne dispose pas d’un ensemble de données d’entraînement avec des exemples de chemins corrects. L’apprentissage non supervisé ne convient pas non plus car il n’y a pas de structure de regroupement à découvrir. Le problème est bien celui d’un agent qui apprend par essais et erreurs, en explorant le labyrinthe, en recevant un retour sous forme de récompenses, et en affinant sa stratégie pour trouver l’itinéraire optimal. On peut également combiner cela avec des algorithmes de recherche classique (A*, Dijkstra) si la carte du labyrinthe est entièrement connue à l’avance, mais si le labyrinthe est inconnu ou partiellement observable, l’apprentissage par renforcement devient indispensable.

#EXERCICE 5

Introduction

J’ai sélectionné trois types de modèles représentatifs : un modèle d’apprentissage supervisé (la régression logistique pour la classification binaire, reprise de l’exercice sur les défauts de prêt), un modèle d’apprentissage non supervisé (l’algorithme K-means pour le clustering, adapté à l’organisation d’une bibliothèque par genres), et un modèle d’apprentissage par renforcement (Q-learning tabulaire pour la navigation d’un robot dans un labyrinthe). Pour chacun, je décris une stratégie d’évaluation complète, puis j’aborde les difficultés et limites propres à chaque catégorie.

1. Modèle supervisé : classification binaire (régression logistique)

Stratégie d’évaluation

Pour évaluer un classifieur supervisé comme la régression logistique dans le cadre de la prédiction des défauts de paiement, je mets en place une stratégie rigoureuse en plusieurs étapes. Tout d’abord, je partitionne les données en trois ensembles : entraînement (70 %), validation (15 %) et test (15 %), avec une stratification pour conserver la proportion de la classe minoritaire (défauts). Ensuite, j’utilise une validation croisée stratifiée à k = 5 plis sur l’ensemble d’entraînement pour obtenir des estimations stables des performances et éviter le sur-ajustement.

Pour le choix des métriques, je ne me fie pas à l’exactitude (accuracy) car elle est trompeuse en présence de déséquilibre de classes. Je retiens donc : le rappel (recall) pour la classe défaut, qui mesure la proportion de vrais défauts correctement identifiés (critère prioritaire pour ne pas laisser passer des mauvais payeurs) ; la précision (precision) pour la classe défaut, qui indique la fiabilité des alertes de défaut ; le score F1 (moyenne harmonique de la précision et du rappel) ; l’aire sous la courbe ROC (AUC-ROC) qui évalue la capacité discriminante du modèle indépendamment du seuil ; et enfin l’aire sous la courbe précision-rappel (AUC-PR) particulièrement adaptée aux datasets déséquilibrés. Je calcule également la matrice de confusion pour visualiser les faux positifs et faux négatifs. En complément, je trace la courbe ROC et je détermine le seuil de décision optimal non pas à 0,5 par défaut mais en maximisant un coût métier (par exemple, coût d’un défaut non détecté vs coût d’un prêt refusé à un bon client).

Difficultés et limites

Plusieurs difficultés apparaissent dans l’évaluation des modèles supervisés. La première est le risque de fuite de données (data leakage) : si des informations futures ou des variables trop corrélées à la cible sont incluses dans les caractéristiques, les performances sur le test seront artificiellement élevées mais le modèle échouera en production. La deuxième difficulté est le déséquilibre de classes : des métriques comme l’accuracy deviennent inutiles, et même l’AUC-ROC peut être trop optimiste si le déséquilibre est extrême (mieux vaut alors se concentrer sur l’AUC-PR ou le rappel à un niveau de précision fixé). La troisième limite est le sur-ajustement (overfitting) : un modèle trop complexe peut obtenir d’excellentes performances sur l’ensemble de validation mais se comporter médiocrement sur de nouvelles données. Enfin, l’évaluation ne capture pas toujours la stabilité temporelle du modèle (concept drift), car la relation entre les caractéristiques et la cible peut évoluer dans le temps (par exemple, en période de crise économique, les comportements de remboursement changent).

2. Modèle non supervisé : clustering (K-means)

Stratégie d’évaluation

L’évaluation d’un modèle de clustering comme K-means est intrinsèquement plus délicate car il n’existe pas de vérité terrain (d’étiquettes). Je combine plusieurs techniques quantitatives et qualitatives. D’abord, le score de silhouette (silhouette score) mesure la cohésion interne des clusters : pour chaque point, il compare la distance moyenne aux points de son propre cluster avec la distance moyenne au cluster le plus proche. Un score proche de 1 indique des clusters bien séparés et denses ; proche de 0, des clusters qui se chevauchent ; négatif, des points mal assignés. Je calcule ce score pour différentes valeurs de k (nombre de clusters) et je retiens celle qui maximise le score moyen.

Ensuite, j’utilise la méthode du coude (elbow method) basée sur l’inertie intra-cluster (somme des carrés des distances entre chaque point et son centroïde). Je trace l’inertie en fonction de k et je cherche le point où la décroissance se ralentit brusquement (le « coude »), indiquant que l’ajout de clusters supplémentaires n’apporte plus de gain significatif. Troisièmement, j’utilise l’indice de Davies-Bouldin qui mesure le rapport entre la dispersion intra-cluster et la séparation inter-clusters : plus la valeur est faible, meilleur est le partitionnement.

Pour une validation plus robuste, je pratique la validation par cluster stability : je ré-échantillonne les données (bootstrap) et je compare les clusters obtenus sur différents sous-ensembles. Des clusters stables sont préférables. Enfin, je procède à une évaluation qualitative : j’examine manuellement quelques livres de chaque cluster pour vérifier leur cohérence thématique (par exemple, le cluster 1 contient effectivement des romans policiers, le cluster 2 des livres d’histoire, etc.). Cela permet de valider l’utilité pratique du regroupement.

Difficultés et limites

L’évaluation du clustering souffre de plusieurs limites fondamentales. La première est l’absence de vérité terrain : on ne peut jamais vraiment savoir si la partition découverte est « correcte ». Le score de silhouette favorise les clusters sphériques et de taille égale, mais certains clusters naturels peuvent être allongés ou de densités variables (auquel cas DBSCAN serait préférable à K-means). La méthode du coude est subjective : le « coude » n’est pas toujours net et peut être ambigu. Ensuite, la validation par stabilité est coûteuse en calcul et dépend du ré-échantillonnage. Une autre limite importante est que la qualité d’un clustering dépend fortement du prétraitement des données (normalisation, réduction de dimension). Enfin, même avec des métriques quantitatives élevées, les clusters peuvent ne pas être exploitables métier : l’évaluation doit donc toujours être complétée par un jugement humain et une validation par des experts du domaine.

3. Modèle d’apprentissage par renforcement : Q-learning (robot dans un labyrinthe)

Stratégie d’évaluation

Pour évaluer un agent d’apprentissage par renforcement comme le Q-learning, je me concentre sur plusieurs aspects complémentaires. Le premier indicateur est la récompense cumulative (cumulative reward) ou le retour épisodique (episodic return) : je fais exécuter l’agent sur un épisode complet (du départ à la sortie du labyrinthe) et je somme les récompenses obtenues. Une récompense cumulative élevée indique que l’agent trouve un chemin court et évite les punitions (murs, temps perdu). Je trace la courbe d’apprentissage (épisode par épisode, la récompense moyenne sur une fenêtre glissante) pour observer la progression.

Le deuxième aspect est la convergence : un bon agent doit atteindre une politique stable où la récompense n’augmente plus significativement. Je mesure donc la variabilité de la récompense sur les derniers épisodes et j’examine la matrice Q pour vérifier si les valeurs Q se stabilisent. Un critère pratique est d’arrêter l’entraînement lorsque l’amélioration moyenne sur 100 épisodes consécutifs est inférieure à un seuil (par exemple 1 %).

Le troisième aspect concerne l’équilibre entre exploration et exploitation. Je mesure le taux d’exploration (par exemple, avec une politique epsilon-greedy, je suis la proportion d’actions aléatoires prises par épisode). Idéalement, ce taux devrait diminuer progressivement : élevé au début pour découvrir le labyrinthe, faible à la fin pour exploiter le chemin optimal. Je calcule également la longueur du chemin obtenu par l’agent (nombre d’étapes pour atteindre la sortie) et je la compare à la longueur du chemin optimal connu (par exemple, calculé par l’algorithme A*). Une politique parfaite atteint ce minimum.

Enfin, je teste la généralisation : si le labyrinthe change légèrement (ajout d’un mur, déplacement de la sortie), dans quelle mesure l’agent s’adapte-t-il ? Pour cela, je mesure la récompense obtenue dans un environnement de test modifié sans réentraînement, puis avec un réentraînement rapide (fine-tuning).

Difficultés et limites

L’évaluation en apprentissage par renforcement présente des défis uniques. Le premier est la variabilité élevée : deux exécutions du même algorithme peuvent donner des résultats très différents à cause de la stochasticité de l’environnement ou des initialisations aléatoires. Il faut donc lancer plusieurs runs indépendants et rapporter des moyennes avec des intervalles de confiance. Le deuxième problème est l’explosion ou l’effondrement des récompenses : certaines politiques peuvent être instables. Le troisième défi est le compromis coût d’évaluation : évaluer une politique peut nécessiter des centaines d’épisodes dans l’environnement réel, ce qui peut être lent ou coûteux (par exemple, pour un robot physique). On utilise souvent des simulateurs, mais l’écart entre simulateur et réalité (sim-to-real gap) est une limite majeure. Ensuite, la convergence n’est pas garantie : Q-learning peut diverger avec des approximants non linéaires comme les réseaux de neurones profonds (DQN). Enfin, la récompense cumulative ne dit rien sur la sécurité ou le comportement intermédiaire : un agent pourrait trouver un chemin court mais dangereux (par exemple, traverser une zone interrompue dans un contexte réel). Il faut parfois ajouter des métriques de sûreté ou de robustesse.

Conclusion générale

L’évaluation des modèles d’apprentissage automatique diffère radicalement selon la catégorie. En supervisé, on dispose de métriques claires et d’une vérité terrain, mais il faut gérer le déséquilibre, le sur-ajustement et la stabilité temporelle. En non supervisé, l’absence de vérité terrain impose des métriques intrinsèques (silhouette, inertie) combinées à une validation qualitative et par stabilité, avec des limites subjectives importantes. En renforcement, on se concentre sur la récompense cumulative et la convergence, mais la variabilité, le coût d’évaluation et l’écart simulateur-réalité sont des obstacles majeurs. Dans tous les cas, une bonne stratégie d’évaluation combine plusieurs indicateurs, des validations croisées ou répliquées, et une interprétation métier des résultats.